In [21]:
import pandas as pd          # main data tool (like Excel in Python)
import numpy as np           # math & number operations
import matplotlib.pyplot as plt  # basic charts
import seaborn as sns        # nicer looking charts
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # show ALL columns
print('All libraries imported successfully!')

df = pd.read_csv('D:\Technocolabs\Pickup Five Cities Datasets\pickup_yt.csv')

print('Number of rows   :', df.shape[0])  # how many records
print('Number of columns:', df.shape[1])  # how many fields

All libraries imported successfully!
Number of rows   : 1048575
Number of columns: 19


In [22]:
df.isnull().sum()

order_id                  0
region_id                 0
city                      0
courier_id                0
accept_time               0
time_window_start         0
time_window_end           0
lng                       0
lat                       0
aoi_id                    0
aoi_type                  0
pickup_time               0
pickup_gps_time      216517
pickup_gps_lng       216517
pickup_gps_lat       216517
accept_gps_time      379864
accept_gps_lng       379864
accept_gps_lat       379864
ds                        0
dtype: int64

In [23]:
gps_cols = ['pickup_gps_lng', 'pickup_gps_lat',
            'accept_gps_lng',  'accept_gps_lat']
df[gps_cols] = df[gps_cols].fillna(0.0)

print(df[gps_cols].isnull().sum())

pickup_gps_lng    0
pickup_gps_lat    0
accept_gps_lng    0
accept_gps_lat    0
dtype: int64


In [24]:


# Step 1 — Reload the original file fresh (resets everything)
df = pd.read_csv('D:\Technocolabs\Pickup Five Cities Datasets\pickup_yt.csv')

# Step 2 — Convert datetime columns (format is MM-DD HH:MM:SS, add year manually)
time_cols = ['accept_time', 'time_window_start', 'time_window_end', 'pickup_time']

for col in time_cols:
    df[col] = pd.to_datetime(df[col],
                              format='%m/%d/%Y %H:%M',
                              errors='coerce')

# Step 3 — Verify it worked
print(df[time_cols].dtypes)
print()
print(df['accept_time'].head())

accept_time          datetime64[ns]
time_window_start    datetime64[ns]
time_window_end      datetime64[ns]
pickup_time          datetime64[ns]
dtype: object

0   2018-01-05 08:16:00
1   2014-01-05 09:17:00
2   2026-05-10 15:30:00
3   2026-05-05 12:47:00
4   2026-05-11 08:02:00
Name: accept_time, dtype: datetime64[ns]


In [25]:
duplicates = df.duplicated(subset=['order_id'])
print('Number of duplicate rows:', duplicates.sum())

Number of duplicate rows: 0


In [26]:
# Example: How many minutes from accept to pickup?
df['accept_to_pickup_min'] = (
    (df['pickup_time'] - df['accept_time']).dt.total_seconds() / 60
)
print(df['accept_to_pickup_min'].describe().round(1))
# This shows average, min, max pickup time in minutes

count     1048575.0
mean        -8003.4
std       2844804.4
min     -52071107.0
25%            59.0
50%           111.0
75%           184.0
max      50880020.0
Name: accept_to_pickup_min, dtype: float64


In [27]:
col = 'accept_to_pickup_min'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = (df[col] < lower) | (df[col] > upper)
print(f'Outlier rows found: {outliers.sum():,}')
print(f'Valid range: {lower:.0f} to {upper:.0f} minutes')

Outlier rows found: 124,038
Valid range: -128 to 372 minutes


In [28]:
df = df[df['accept_to_pickup_min'] >= 0].reset_index(drop=True)
print('Rows after removing negatives:', len(df))

Rows after removing negatives: 1044791


In [29]:
df['accept_to_pickup_min'] = df['accept_to_pickup_min'].clip(upper=upper)
print('Max duration after clipping:', df['accept_to_pickup_min'].max())

Max duration after clipping: 371.5


In [30]:
df['accept_hour']  = df['accept_time'].dt.hour    # 0 to 23
df['accept_day']   = df['accept_time'].dt.day     # 1 to 31
df['accept_month'] = df['accept_time'].dt.month   # 1 to 12
print(df[['accept_time', 'accept_hour', 'accept_day', 'accept_month']].head(3))

          accept_time  accept_hour  accept_day  accept_month
0 2018-01-05 08:16:00            8           5             1
1 2014-01-05 09:17:00            9           5             1
2 2026-05-10 15:30:00           15          10             5


In [31]:
def time_of_day(hour):
    if   0 <= hour < 6:   return 'Night'
    elif 6 <= hour < 12:  return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else:                  return 'Night'
df['time_of_day'] = df['accept_hour'].apply(time_of_day)
print(df['time_of_day'].value_counts())
print(df['time_of_day'].value_counts().sum())

time_of_day
Morning      817819
Afternoon    225993
Evening         881
Night            98
Name: count, dtype: int64
1044791


In [32]:
df['window_duration_min'] = (
    (df['time_window_end'] - df['time_window_start']).dt.total_seconds() / 60
)
print(df['window_duration_min'].value_counts().head())


window_duration_min
120.0    1037674
899.0       1912
561.0         18
519.0         18
622.0         17
Name: count, dtype: int64


In [33]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min',
            'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month',
            'time_of_day', 'window_duration_min']
print(new_cols)

Final dataset shape: (1044791, 25)

Missing values left:
pickup_gps_time    215770
pickup_gps_lng     215770
pickup_gps_lat     215770
accept_gps_time    376080
accept_gps_lng     376080
accept_gps_lat     376080
dtype: int64

New columns created:
['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min', 'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'window_duration_min']


In [34]:
# Check all text columns and their unique values
text_cols = df.select_dtypes(include='object').columns.tolist()
print('Text columns:', text_cols)
print()

for col in text_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()
    

Text columns: ['city', 'pickup_gps_time', 'accept_gps_time', 'time_of_day']

--- city ---
city
Yantai    1044791
Name: count, dtype: int64

--- pickup_gps_time ---
pickup_gps_time
01/06/2021 09:13    33
01/06/2026 09:06    33
01/06/2021 09:02    32
07/11/2026 09:12    32
01/06/2020 09:32    31
                    ..
10/08/2026 19:04     1
10/04/2026 08:05     1
01/06/2022 07:35     1
01/10/2014 19:48     1
01/06/2015 15:13     1
Name: count, Length: 122908, dtype: int64

--- accept_gps_time ---
accept_gps_time
01/06/2020 07:37    87
01/06/2023 07:40    84
01/06/2023 07:50    79
01/06/2021 07:44    77
01/06/2027 07:42    76
                    ..
10/04/2026 14:03     1
01/10/2024 16:10     1
08/04/2026 07:13     1
01/06/2013 16:01     1
01/10/2027 14:12     1
Name: count, Length: 101377, dtype: int64

--- time_of_day ---
time_of_day
Morning      817819
Afternoon    225993
Evening         881
Night            98
Name: count, dtype: int64



In [35]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min',
            'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month',
            'time_of_day', 'window_duration_min']
print(new_cols)

Final dataset shape: (1044791, 25)

Missing values left:
pickup_gps_time    215770
pickup_gps_lng     215770
pickup_gps_lat     215770
accept_gps_time    376080
accept_gps_lng     376080
accept_gps_lat     376080
dtype: int64

New columns created:
['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min', 'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'window_duration_min']


In [36]:
# Check all text columns and their unique values
text_cols = df.select_dtypes(include='object').columns.tolist()
print('Text columns:', text_cols)
print()

for col in text_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()
    

Text columns: ['city', 'pickup_gps_time', 'accept_gps_time', 'time_of_day']

--- city ---
city
Yantai    1044791
Name: count, dtype: int64

--- pickup_gps_time ---
pickup_gps_time
01/06/2021 09:13    33
01/06/2026 09:06    33
01/06/2021 09:02    32
07/11/2026 09:12    32
01/06/2020 09:32    31
                    ..
10/08/2026 19:04     1
10/04/2026 08:05     1
01/06/2022 07:35     1
01/10/2014 19:48     1
01/06/2015 15:13     1
Name: count, Length: 122908, dtype: int64

--- accept_gps_time ---
accept_gps_time
01/06/2020 07:37    87
01/06/2023 07:40    84
01/06/2023 07:50    79
01/06/2021 07:44    77
01/06/2027 07:42    76
                    ..
10/04/2026 14:03     1
01/10/2024 16:10     1
08/04/2026 07:13     1
01/06/2013 16:01     1
01/10/2027 14:12     1
Name: count, Length: 101377, dtype: int64

--- time_of_day ---
time_of_day
Morning      817819
Afternoon    225993
Evening         881
Night            98
Name: count, dtype: int64



In [37]:
for col in text_cols:
    df[col] = df[col].str.strip()        # remove spaces before/after
    df[col] = df[col].str.title()        # Capitalize First Letter
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)  # remove double spaces

print('Text columns cleaned!')
print(df['city'].value_counts())

Text columns cleaned!
city
Yantai    1044791
Name: count, dtype: int64


In [38]:
# Cell 12b — Save cleaned data to a new CSV file
df.to_csv('pickup_yt_cleaned.csv', index=False)
print('File saved as: pickup_yt_cleaned.csv')
print(f'Total rows saved: {len(df):,}')

File saved as: pickup_yt_cleaned.csv
Total rows saved: 1,044,791


In [45]:
print(df['city'].value_counts())
print()
print("Total rows:", len(df))
print("Unique city names:", df['city'].nunique())

city
Yantai    1044791
Name: count, dtype: int64

Total rows: 1044791
Unique city names: 1
